# Taller 2 — Reglas de Validación y Control de Versiones con Git

**Solución de referencia para el instructor**  
Caso: **TiendaNova** · Datasets: `clientes.csv`, `pedidos.csv`

Esta notebook implementa 5 reglas de negocio como funciones reutilizables, genera un resumen de cumplimiento y guarda evidencia de errores de integridad referencial.


## 1. Preparación

En Google Colab, sube los archivos `clientes.csv` y `pedidos.csv` al entorno antes de ejecutar la siguiente celda.


In [1]:
import pandas as pd

clientes = pd.read_csv("clientes.csv")
pedidos = pd.read_csv("pedidos.csv")

print("Clientes:", clientes.shape)
print("Pedidos:", pedidos.shape)


Clientes: (193, 10)
Pedidos: (266, 7)


## 2. Exploración inicial de los datasets

Antes de aplicar las reglas de negocio, revisamos la estructura general de `clientes` y `pedidos`: primeras filas, tipos de dato, nulos, duplicados y valores categóricos.

In [2]:
print("Vista previa de clientes:")
display(clientes.head())

print("\nVista previa de pedidos:")
display(pedidos.head())

Vista previa de clientes:


,id_cliente,nombre,email,telefono,fecha_nacimiento,ciudad,fecha_registro,monto_compra_total,num_compras,canal_registro
0,28,Raul Vargas,raul.vargas@correo.com,928024248,1971-02-04,Trujillo,18/03/2026,1261.95,20.0,App
1,33,Sergio Perez,sergio.perez@correo.com,934073380,2002-06-26,Cusco,26/11/2024,3895.08,8.0,Tienda fisica
2,40,Sofia Vargas,sofia.vargas@correo.com,948508907,1968-07-26,LIMA,20/11/2025,1484.01,15.0,Telefono
3,70,Jorge Perez,jorge.perez@correo.com,+51 92325893,1955-07-15,Piura,20/08/2026,1346.1,8.0,Tienda fisica
4,20,Rosa Sanchez,rosa.sanchez@correo.com,928814949,1982-03-09,piura,08/02/2024,2021.93,18.0,Web



Vista previa de pedidos:


,id_pedido,id_cliente,id_producto,cantidad,fecha_pedido,fecha_entrega,estado
0,130,141,10,1,2026-10-17,2026-10-18,En camino
1,244,76,24,4,2026-08-20,2026-08-23,Pendiente
2,230,70,40,5,2026-07-17,2026-07-19,Pendiente
3,104,24,37,1,2026-04-26,2026-04-28,Entregado
4,95,142,26,5,2026-06-04,2026-06-10,Pendiente


In [3]:
print("=== Tipos de dato y nulos: clientes ===")
clientes.info()

print("\n=== Tipos de dato y nulos: pedidos ===")
pedidos.info()

=== Tipos de dato y nulos: clientes ===
<class 'pandas.DataFrame'>
RangeIndex: 193 entries, 0 to 192
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_cliente          193 non-null    int64  
 1   nombre              193 non-null    str    
 2   email               185 non-null    str    
 3   telefono            185 non-null    str    
 4   fecha_nacimiento    193 non-null    str    
 5   ciudad              183 non-null    str    
 6   fecha_registro      193 non-null    str    
 7   monto_compra_total  193 non-null    str    
 8   num_compras         186 non-null    float64
 9   canal_registro      193 non-null    str    
dtypes: float64(1), int64(1), str(8)
memory usage: 15.2 KB

=== Tipos de dato y nulos: pedidos ===
<class 'pandas.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0

In [4]:
print("Estadísticas descriptivas: clientes")
display(clientes.describe(include="all"))

print("\nEstadísticas descriptivas: pedidos")
display(pedidos.describe(include="all"))

Estadísticas descriptivas: clientes


,id_cliente,nombre,email,telefono,fecha_nacimiento,ciudad,fecha_registro,monto_compra_total,num_compras,canal_registro
count,193.000000,193,185,185,193,183,193,193,186.000000,193
unique,NaN,159,152,175,175,17,173,181,NaN,4
top,NaN,Rosa Sanchez,rosa.sanchez@correo.com,928814949,2031-01-15,Piura,20/08/2026,1346.1,NaN,Tienda fisica
freq,NaN,3,3,2,3,19,2,2,NaN,54
mean,89.792746,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.268817,NaN
std,51.961510,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.351617,NaN
min,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,NaN
25%,45.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.000000,NaN
50%,90.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.000000,NaN
75%,134.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.000000,NaN



Estadísticas descriptivas: pedidos


,id_pedido,id_cliente,id_producto,cantidad,fecha_pedido,fecha_entrega,estado
count,266.000000,266.000000,266.000000,266.000000,266,266,266
unique,NaN,NaN,NaN,NaN,174,151,4
top,NaN,NaN,NaN,NaN,2026-07-17,2026-01-01,Entregado
freq,NaN,NaN,NaN,NaN,4,11,84
mean,131.285714,426.812030,219.684211,3.375940,NaN,NaN,NaN
std,74.912333,1795.460158,1319.341719,2.208394,NaN,NaN,NaN
min,1.000000,1.000000,1.000000,-6.000000,NaN,NaN,NaN
25%,67.250000,45.250000,10.000000,2.000000,NaN,NaN,NaN
50%,132.500000,99.500000,20.000000,3.500000,NaN,NaN,NaN
75%,195.750000,146.750000,29.750000,5.000000,NaN,NaN,NaN


In [5]:
print("=== Filas duplicadas ===")
print(f"Clientes: {clientes.duplicated().sum()} duplicados")
print(f"Pedidos:  {pedidos.duplicated().sum()} duplicados")

=== Filas duplicadas ===
Clientes: 5 duplicados
Pedidos:  6 duplicados


In [6]:
print("=== Valores únicos en columnas categóricas ===")
print("Ciudades (clientes):", sorted(clientes["ciudad"].dropna().unique()))
print("Canales de registro (clientes):", sorted(clientes["canal_registro"].dropna().unique()))
print("Estados de pedido (pedidos):", sorted(pedidos["estado"].dropna().unique()))

=== Valores únicos en columnas categóricas ===
Ciudades (clientes): [' Lima', 'AREQUIPA', 'Arequipa', 'Chiclayo', 'Cusco', 'Cuzco', 'LIMA', 'Lima', 'Lima ', 'Piura', 'Trujillo', 'arequipa', 'chiclayo', 'cusco', 'lima', 'piura', 'trujillo']
Canales de registro (clientes): ['App', 'Telefono', 'Tienda fisica', 'Web']
Estados de pedido (pedidos): ['Cancelado', 'En camino', 'Entregado', 'Pendiente']


## 3. Catálogo de reglas de negocio

Las siguientes funciones implementan las reglas R001–R006.

In [7]:
def regla_email_valido(df):
    """R001 — Todo email debe contener '@' y no estar vacío."""
    return df[df["email"].isnull() | ~df["email"].astype(str).str.contains("@", na=False)]


def regla_cantidad_positiva(df):
    """R002 — La cantidad de un pedido debe ser mayor a cero."""
    return df[df["cantidad"] <= 0]


def regla_fecha_entrega_posterior(df):
    """R003 — La fecha de entrega no puede ser anterior a la fecha de pedido (consistencia)."""
    d = df.copy()
    d["fecha_pedido"] = pd.to_datetime(d["fecha_pedido"])
    d["fecha_entrega"] = pd.to_datetime(d["fecha_entrega"])
    return d[d["fecha_entrega"] < d["fecha_pedido"]]


def regla_integridad_cliente(pedidos_df, clientes_df):
    """R004 — Todo id_cliente en pedidos debe existir en clientes (integridad referencial)."""
    ids_validos = set(clientes_df["id_cliente"])
    return pedidos_df[~pedidos_df["id_cliente"].isin(ids_validos)]


def regla_no_duplicados_pedido(df):
    """R005 — No deben existir pedidos exactamente duplicados."""
    return df[df.duplicated()]


def regla_ciudad_normalizada(df):
    """R006 — El nombre de la ciudad no debe tener espacios extra ni variar en
    mayúsculas/minúsculas; debe estar en formato Título (ej. 'Lima')."""
    ciudad_normalizada = df["ciudad"].astype(str).str.strip().str.title()
    return df[df["ciudad"].notna() & (df["ciudad"].astype(str) != ciudad_normalizada)]

## 4. Ejecución consolidada de las reglas

In [8]:
def ejecutar_todas_las_reglas(clientes_df, pedidos_df):
    """Corre las 6 reglas y devuelve un resumen de cumplimiento."""
    resultados = {
        "R001 Email válido (clientes)": regla_email_valido(clientes_df),
        "R002 Cantidad positiva (pedidos)": regla_cantidad_positiva(pedidos_df),
        "R003 Fecha entrega >= fecha pedido (pedidos)": regla_fecha_entrega_posterior(pedidos_df),
        "R004 Integridad referencial id_cliente (pedidos -> clientes)": regla_integridad_cliente(pedidos_df, clientes_df),
        "R005 Pedidos sin duplicados exactos": regla_no_duplicados_pedido(pedidos_df),
        "R006 Ciudad normalizada (clientes)": regla_ciudad_normalizada(clientes_df),
    }

    resumen = {nombre: len(df_errores) for nombre, df_errores in resultados.items()}
    return resultados, resumen

In [9]:
resultados, resumen = ejecutar_todas_las_reglas(clientes, pedidos)

print("=== Resumen de cumplimiento de reglas ===")
for nombre, n_errores in resumen.items():
    print(f"{nombre}: {n_errores} incumplimientos")


=== Resumen de cumplimiento de reglas ===
R001 Email válido (clientes): 12 incumplimientos
R002 Cantidad positiva (pedidos): 9 incumplimientos
R003 Fecha entrega >= fecha pedido (pedidos): 11 incumplimientos
R004 Integridad referencial id_cliente (pedidos -> clientes): 9 incumplimientos
R005 Pedidos sin duplicados exactos: 6 incumplimientos
R006 Ciudad normalizada (clientes): 101 incumplimientos


## 5. Evidencia de errores de integridad referencial

In [10]:
resultados["R004 Integridad referencial id_cliente (pedidos -> clientes)"].to_csv(
    "errores_integridad_referencial.csv", index=False
)

print("Detalle de errores de integridad referencial guardado en errores_integridad_referencial.csv")


Detalle de errores de integridad referencial guardado en errores_integridad_referencial.csv


## 6. Guía rápida de Git

Estos comandos se ejecutan en una **terminal**, no en Python:

```bash
git init
git status
git add Taller2_Solucion_Instructor.py
git commit -m "Agrega reglas de validacion R001-R005 para clientes y pedidos"
git log --oneline

# Opcional: conectar con GitHub
git remote add origin https://github.com/<usuario>/<repositorio>.git
git push -u origin main
```

**Entregable del Taller 2:** este trabajo, un repositorio Git local con al menos 2 commits y una captura de pantalla de `git log --oneline`.